In [1]:
pip install psycopg2-binary python-dotenv bcrypt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import psycopg2
from psycopg2.extras import RealDictCursor
import os
import bcrypt
from dataclasses import dataclass

# Конфигурация базы данных
@dataclass
class DatabaseConfig:
    host: str
    port: int
    database: str
    user: str
    password: str

def get_db_config() -> DatabaseConfig:
    return DatabaseConfig(
        host=os.getenv('DB_HOST', 'localhost'),
        port=int(os.getenv('DB_PORT', '5432')),
        database=os.getenv('DB_NAME', 'postgres'),
        user=os.getenv('DB_USER', 'postgres'),
        password=os.getenv('DB_PASSWORD', 'password')
    )

# Функции для работы с паролями
def hash_password(password: str) -> str:
    """Хеширование пароля"""
    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(password.encode('utf-8'), salt)
    return hashed.decode('utf-8')

def verify_password(plain_password: str, hashed_password: str) -> bool:
    """Проверка пароля"""
    try:
        return bcrypt.checkpw(
            plain_password.encode('utf-8'), 
            hashed_password.encode('utf-8')
        )
    except Exception:
        return False

# Основной класс для работы с БД
class DatabaseManager:
    def __init__(self):
        self.config = get_db_config()
    
    def get_connection(self):
        """Получение подключения к БД"""
        try:
            return psycopg2.connect(
                host=self.config.host,
                port=self.config.port,
                database=self.config.database,
                user=self.config.user,
                password=self.config.password,
                cursor_factory=RealDictCursor
            )
        except Exception as e:
            print(f"Ошибка подключения к БД: {e}")
            return None
    
    def create_tables(self):
        """Создание таблиц"""
        conn = self.get_connection()
        if conn is None:
            return False
            
        try:
            with conn.cursor() as cur:
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS users (
                        id SERIAL PRIMARY KEY,
                        username VARCHAR(50) UNIQUE NOT NULL,
                        email VARCHAR(100) UNIQUE NOT NULL,
                        password_hash VARCHAR(255) NOT NULL,
                        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                    )
                """)
            conn.commit()
            print("Таблицы успешно созданы")
            return True
        except Exception as e:
            print(f"Ошибка при создании таблиц: {e}")
            return False
        finally:
            conn.close()
    
    def create_user(self, username: str, email: str, password: str) -> bool:
        """Создание пользователя с хешированным паролем"""
        password_hash = hash_password(password)
        
        conn = self.get_connection()
        if conn is None:
            return False
            
        try:
            with conn.cursor() as cur:
                cur.execute(
                    """
                    INSERT INTO users (username, email, password_hash)
                    VALUES (%s, %s, %s)
                    """,
                    (username, email, password_hash)
                )
            conn.commit()
            print(f"Пользователь {username} успешно создан")
            return True
        except psycopg2.IntegrityError:
            print(f"Ошибка: пользователь с таким именем или email уже существует")
            return False
        except Exception as e:
            print(f"Ошибка при создании пользователя: {e}")
            return False
        finally:
            conn.close()
    
    def verify_user(self, username: str, password: str) -> bool:
        """Проверка пользователя"""
        conn = self.get_connection()
        if conn is None:
            return False
            
        try:
            with conn.cursor() as cur:
                cur.execute(
                    "SELECT password_hash FROM users WHERE username = %s",
                    (username,)
                )
                result = cur.fetchone()
                
                if result and verify_password(password, result['password_hash']):
                    print("Пароль верный")
                    return True
                else:
                    print("Неверное имя пользователя или пароль")
                    return False
        except Exception as e:
            print(f"Ошибка при проверке пользователя: {e}")
            return False
        finally:
            conn.close()
    
    def get_all_users(self):
        """Получение всех пользователей (для тестирования)"""
        conn = self.get_connection()
        if conn is None:
            return []
            
        try:
            with conn.cursor() as cur:
                cur.execute("SELECT id, username, email, created_at FROM users")
                users = cur.fetchall()
                return users
        except Exception as e:
            print(f"Ошибка при получении пользователей: {e}")
            return []
        finally:
            conn.close()
    
    def drop_tables(self):
        """Удаление таблиц (для тестирования)"""
        conn = self.get_connection()
        if conn is None:
            return False
            
        try:
            with conn.cursor() as cur:
                cur.execute("DROP TABLE IF EXISTS users")
            conn.commit()
            print("Таблицы удалены")
            return True
        except Exception as e:
            print(f"Ошибка при удалении таблиц: {e}")
            return False
        finally:
            conn.close()

# Основная функция
def main():
    # Настройки БД (можно изменить под вашу конфигурацию)
    db_config = {
        'DB_HOST': 'localhost',
        'DB_PORT': '5432',
        'DB_NAME': 'DB6',
        'DB_USER': 'postgres',
        'DB_PASSWORD': '2238'  # замените на ваш пароль
    }
    
    # Устанавливаем переменные окружения
    for key, value in db_config.items():
        os.environ[key] = value
    
    db = DatabaseManager()
    
    # Проверка подключения
    print("Проверка подключения к базе данных...")
    conn = db.get_connection()
    if conn:
        print("Подключение к БД успешно")
        conn.close()
    else:
        print("Не удалось подключиться к БД")
        return
    
    # Создание таблиц
    db.create_tables()
    
    # Демонстрация работы
    print("Демонстрация работы системы:")
    
    # Создаем тестовых пользователей
    test_users = [
        ("alex", "alex@example.com", "password123"),
        ("maria", "maria@example.com", "securepass")
    ]
    
    for username, email, password in test_users:
        db.create_user(username, email, password)
    
    # Проверяем аутентификацию
    print("Тест аутентификации:")
    db.verify_user("alex", "password123")  # Верный пароль
    db.verify_user("alex", "wrongpass")    # Неверный пароль
    
    # Показываем всех пользователей
    print("Все пользователи в системе:")
    users = db.get_all_users()
    for user in users:
        print(f"   - {user['username']} ({user['email']})")
    
    # Интерактивный режим
    print("Интерактивный режим:")
    while True:
        print("\n" + "="*50)
        print("1. Создать пользователя")
        print("2. Проверить пользователя")
        print("3. Показать всех пользователей")
        print("4. Очистить базу данных")
        print("5. Выйти")
        print("="*50)
        
        choice = input("Выберите действие: ").strip()
        
        if choice == '1':
            username = input("Введите имя пользователя: ").strip()
            email = input("Введите email: ").strip()
            password = input("Введите пароль: ").strip()
            if username and email and password:
                db.create_user(username, email, password)
            else:
                print("Все поля должны быть заполнены")
        
        elif choice == '2':
            username = input("Введите имя пользователя: ").strip()
            password = input("Введите пароль: ").strip()
            if username and password:
                db.verify_user(username, password)
            else:
                print("Все поля должны быть заполнены")
        
        elif choice == '3':
            users = db.get_all_users()
            print("\n📋 Список пользователей:")
            if users:
                for user in users:
                    print(f"   ID: {user['id']}, Username: {user['username']}, Email: {user['email']}")
            else:
                print("   Пользователей нет")
        
        elif choice == '4':
            confirm = input("Вы уверены? Все данные будут удалены! (y/n): ")
            if confirm.lower() == 'y':
                db.drop_tables()
                db.create_tables()
        
        elif choice == '5':
            print("Выход из программы...")
            break
        
        else:
            print("Неверный выбор")

if __name__ == "__main__":
    main()

Проверка подключения к базе данных...
Подключение к БД успешно
Таблицы успешно созданы
Демонстрация работы системы:
Ошибка: пользователь с таким именем или email уже существует
Ошибка: пользователь с таким именем или email уже существует
Тест аутентификации:
Пароль верный
Неверное имя пользователя или пароль
Все пользователи в системе:
   - alex (alex@example.com)
   - maria (maria@example.com)
   - Angel (m.a@mail.ru)
   - Q (q.q@mail.ru)
Интерактивный режим:

1. Создать пользователя
2. Проверить пользователя
3. Показать всех пользователей
4. Очистить базу данных
5. Выйти
Выберите действие: 1
Введите имя пользователя: alex1
Введите email: a/a
Введите пароль: 12345
Пользователь alex1 успешно создан

1. Создать пользователя
2. Проверить пользователя
3. Показать всех пользователей
4. Очистить базу данных
5. Выйти
Выберите действие: 2
Введите имя пользователя: alex1
Введите пароль: 12345
Пароль верный

1. Создать пользователя
2. Проверить пользователя
3. Показать всех пользователей
4. Очи